In [1]:
# Set up libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.tree import plot_tree
from sklearn.pipeline import Pipeline
import numpy as np
import matplotlib.pyplot as plt
import joblib

In [2]:
# Read the clean dataset as DataFrame
path = r"AmesHousingClean.csv"
df = pd.read_csv(path)

In [25]:
# Split the data into features and target and into training and testing datasets
X = df.drop(columns=['SalePrice']).to_numpy()
y = df['SalePrice'].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [26]:
# Scale using Standard Scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
# Do grid-serach to find the best hyperparameters to plug in Random Forest Regressor
param_grid = {
    'n_estimators': [100, 150, 200, 250, 300],
    'max_features': ['sqrt', 'log2', 1.0],
    'max_depth': [10, 20, 30, None],
    'min_samples_leaf': [1, 2, 4],
    'min_samples_split': [2, 5, 10],
}

random_search = RandomizedSearchCV(RandomForestRegressor(random_state=42), 
                                   param_distributions=param_grid, 
                                   n_iter=30,
                                   cv=5,
                                   scoring='neg_mean_absolute_error',
                                   n_jobs=-2, 
                                   random_state=42)
random_search.fit(X_train_scaled, y_train)
print('Best Parameters', random_search.best_params_)


KeyboardInterrupt



In [ ]:
    # n_estimators=215,
    #                                      max_depth=9,
    #                                      max_features='sqrt',
    #                                      min_samples_leaf=1,
    #                                      min_samples_split=2,

In [28]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("forest", RandomForestRegressor(n_estimators=215, max_features='sqrt', min_samples_split=2, min_samples_leaf=1, max_depth=10, random_state=42)),
])

pipe.fit(X_train, y_train)
y_train_pred_rf = pipe.predict(X_train)
y_pred_rf = (pipe.predict(X_test))
joblib.dump(pipe, "random_forest_pipeline.pkl")


mae_train = mean_absolute_error(y_train, y_train_pred_rf)
mae_test = mean_absolute_error(y_test, y_pred_rf)
mape_train = mean_absolute_percentage_error(y_train, y_train_pred_rf) * 100
mape_test = mean_absolute_percentage_error(y_test, y_pred_rf) * 100
r2_train = r2_score(y_train, y_train_pred_rf)
r2_test = r2_score(y_test, y_pred_rf)

print(f'MAE (training): {mae_train}')
print(f'MAE (testing): {mae_test}')
print(f'MAPE (training): {mape_train:.4f}%')
print(f'MAPE (testing): {mape_test:.4f}%')
print(f'R-Squared score (training): {r2_train}')
print(f'R-Squared score (testing): {r2_test}')

MAE (training): 10641.172862174146
MAE (testing): 16789.550111063323
MAPE (training): 6.3330%
MAPE (testing): 9.6779%
R-Squared score (training): 0.966775892313526
R-Squared score (testing): 0.8846276947096401


In [ ]:
# The data is outdated so we need an inflation factor so that the model also works when dealing with new data
inflation_factor = 1.867

In [36]:
new_data = pd.DataFrame({
        'Overall Qual': [7],
        'Lot Area': [2405],
        'Sqft': [1748],
        'Has Basement': [1],
        'Garage Cars': [2],
        'Year Built': [2019],
        'Total Rooms': [14],
        'Bedrooms': [4],
        'Bathrooms': [4],
        'Fireplaces': [np.nan],
    })

pipe.predict(new_data) * 1.867

C:\Users\sungj\Documents\GitHub\Atlantic-Project\.venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


array([338195.45915245])